# Session 03 Practice: Road Network Graphs & Dijkstra's Shortest Path Search
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 2 | Session 3**  

---

### 🎯 Learning Objectives
1. Construct directed road network graphs $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ with spatial node coordinates and speed limits.
2. Implement **Dijkstra's Shortest Path Algorithm** from scratch using a min-heap priority queue (`heapq`).
3. Formulate edge travel time weights: $w_{uv} = \frac{d(u, v)}{v_{\text{speed}}}$.
4. Visualize road networks, explored frontiers, and optimal route telemetry using **Plotly**.

---

### 📐 Mathematical Formulation

Given a directed graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$:
- Cost to come: $g(v) = \min_{u \in \text{pred}(v)} \left(g(u) + w(u, v)\right)$
- Edge weight: $w(u, v) = \frac{\sqrt{(x_v - x_u)^2 + (y_v - y_u)^2}}{v_{\text{limit}}(u, v)}$


In [ ]:
import heapq
import math
import networkx as nx
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. Road Network Construction

In [ ]:
def create_sample_road_network() -> nx.DiGraph:
    G = nx.DiGraph()
    
    nodes = {
        0: (0.0, 0.0),
        1: (100.0, 0.0),
        2: (250.0, 0.0),
        3: (0.0, 100.0),
        4: (100.0, 100.0),
        5: (250.0, 100.0),
        6: (0.0, 250.0),
        7: (120.0, 250.0),
        8: (250.0, 250.0),
    }
    for node_id, pos in nodes.items():
        G.add_node(node_id, pos=pos)
        
    # Directed road segments (u, v, speed_limit_m/s)
    # 50 km/h ~ 13.8 m/s, 80 km/h ~ 22.2 m/s
    edges = [
        (0, 1, 13.8), (1, 2, 22.2), (1, 0, 13.8), (2, 1, 22.2),
        (3, 4, 13.8), (4, 5, 13.8), (4, 3, 13.8), (5, 4, 13.8),
        (6, 7, 13.8), (7, 8, 22.2), (7, 6, 13.8), (8, 7, 22.2),
        (0, 3, 13.8), (3, 6, 13.8), (3, 0, 13.8), (6, 3, 13.8),
        (1, 4, 13.8), (4, 7, 13.8), (4, 1, 13.8), (7, 4, 13.8),
        (2, 5, 22.2), (5, 8, 22.2), (5, 2, 22.2), (8, 5, 22.2),
        (4, 8, 13.8), # diagonal avenue
    ]
    
    for u, v, speed in edges:
        pos_u = nodes[u]
        pos_v = nodes[v]
        dist = math.hypot(pos_v[0] - pos_u[0], pos_v[1] - pos_u[1])
        travel_time = dist / speed
        G.add_edge(u, v, length=dist, speed=speed, weight=travel_time)
        
    return G

G = create_sample_road_network()
print(f"Road network created: {len(G.nodes)} intersections, {len(G.edges)} directed road segments.")


## 2. Dijkstra Algorithm Implementation

In [ ]:
def dijkstra_shortest_path(graph: nx.DiGraph, start_node: int, goal_node: int) -> tuple[list[int], float, int, list[int]]:
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start_node] = 0.0
    parent = {node: None for node in graph.nodes}
    
    pq = [(0.0, start_node)]
    visited = set()
    explored_order = []
    
    while pq:
        current_g, u = heapq.heappop(pq)
        
        if u in visited:
            continue
        visited.add(u)
        explored_order.append(u)
        
        if u == goal_node:
            path = []
            curr = goal_node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return path, current_g, len(explored_order), explored_order
            
        for v in graph.neighbors(u):
            weight = graph[u][v]['weight']
            tentative_g = current_g + weight
            if tentative_g < g_score[v]:
                g_score[v] = tentative_g
                parent[v] = u
                heapq.heappush(pq, (tentative_g, v))
                
    return [], float('inf'), len(explored_order), explored_order

path, cost, explored, order = dijkstra_shortest_path(G, 0, 8)
print(f"Optimal route from Node 0 to Node 8: {path}")
print(f"Total estimated travel time: {cost:.2f} seconds ({explored} nodes explored)")


## 3. Interactive Route Visualizer & Segment Telemetry Plots

In [ ]:
def plot_dijkstra_dashboard(graph: nx.DiGraph, path: list[int], explored_order: list[int]):
    edge_x = []
    edge_y = []
    for u, v in graph.edges():
        x0, y0 = graph.nodes[u]['pos']
        x1, y1 = graph.nodes[v]['pos']
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        
    node_x = [graph.nodes[n]['pos'][0] for n in graph.nodes]
    node_y = [graph.nodes[n]['pos'][1] for n in graph.nodes]
    node_text = [f"Node {n}<br>Explored Step: {explored_order.index(n) if n in explored_order else 'Unvisited'}" for n in graph.nodes]
    
    fig = go.Figure()
    
    # 1. Base Road Edges
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y,
        mode='lines',
        line=dict(width=2, color='#b0bec5'),
        hoverinfo='none',
        name='Road Network'
    ))
    
    # 2. Intersections
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        marker=dict(size=14, color='#90caf9', line=dict(width=2, color='#1565c0')),
        text=[f"N{n}" for n in graph.nodes],
        textposition="top center",
        hovertext=node_text,
        hoverinfo="text",
        name='Intersections'
    ))
    
    # 3. Optimal Path
    if path:
        path_x = [graph.nodes[n]['pos'][0] for n in path]
        path_y = [graph.nodes[n]['pos'][1] for n in path]
        fig.add_trace(go.Scatter(
            x=path_x, y=path_y,
            mode='lines+markers',
            line=dict(width=6, color='#d32f2f'),
            marker=dict(size=16, color='#ffd600', symbol="star"),
            name='Dijkstra Optimal Route'
        ))
        
    fig.update_layout(
        title="Mission Planning: Road Network Graph & Dijkstra Optimal Shortest Path",
        xaxis_title="Global X (m)",
        yaxis_title="Global Y (m)",
        showlegend=True,
        width=850,
        height=650
    )
    fig.show()

plot_dijkstra_dashboard(G, path, order)


In [ ]:
# Segment Breakdown Plot (Time & Distance per Segment)
segment_names = []
segment_times = []
segment_lens = []

for i in range(len(path) - 1):
    u, v = path[i], path[i+1]
    segment_names.append(f"N{u} → N{v}")
    segment_times.append(G[u][v]["weight"])
    segment_lens.append(G[u][v]["length"])

fig_seg = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Segment Travel Time (seconds)", "Segment Distance (meters)")
)

fig_seg.add_trace(
    go.Bar(x=segment_names, y=segment_times, marker_color="#1f77b4", name="Travel Time (s)"),
    row=1, col=1
)
fig_seg.add_trace(
    go.Bar(x=segment_names, y=segment_lens, marker_color="#ff7f0e", name="Distance (m)"),
    row=1, col=2
)

fig_seg.update_layout(title="Optimal Route Segment Profile", height=400, showlegend=False)
fig_seg.show()


## 4. Verification Tests

In [ ]:
nx_path = nx.dijkstra_path(G, 0, 8, weight="weight")
nx_cost = nx.dijkstra_path_length(G, 0, 8, weight="weight")

assert path == nx_path, f"Computed path {path} does not match NetworkX {nx_path}!"
assert math.isclose(cost, nx_cost, rel_tol=1e-4), f"Computed cost {cost} does not match NetworkX {nx_cost}!"

print("✅ All Dijkstra mission planning tests passed successfully!")
